# Beacon WebSocket SQL Test Client

This notebook tests `/api/ws/sql` with either Basic auth handshake or ticket-based auth.

## Requirements
- `BEACON_WS_SQL_ENABLE=true`
- `BEACON_ENABLE_SQL=true`
- Python packages: `websockets`, `httpx`, `pyarrow`

In [102]:
# Optional: install dependencies in the active kernel
%pip install websockets httpx pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [118]:
import asyncio
import base64
import json
import uuid

import httpx
import pandas as pd
import pyarrow.ipc as ipc
import websockets

In [143]:
# Configuration
HOST = "127.0.0.1"
PORT = 5001
USE_TLS = False
AUTH_MODE = "basic"  # "ticket" or "basic"
USERNAME = "beacon-admin"
PASSWORD = "beacon-password"
SQL = "SELECT 1"
TIMEOUT_SECS = 30.0
WS_PING_INTERVAL_SECS = 20.0
WS_PING_TIMEOUT_SECS = None  # None disables client-side ping timeout

In [144]:
def build_basic_auth_header(username: str, password: str) -> dict:
    token = base64.b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
    return {"Authorization": f"Basic {token}"}


async def fetch_ws_ticket(http_base: str, username: str, password: str, timeout: float) -> str:
    headers = build_basic_auth_header(username, password)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.post(f"{http_base}/api/ws/sql-ticket", headers=headers)
        response.raise_for_status()
        payload = response.json()
        return payload["ticket"]


def decode_arrow_stream_to_dataframes(binary_payload: bytes):
    row_count = 0
    dataframes = []
    reader = ipc.open_stream(binary_payload)
    for batch in reader:
        row_count += batch.num_rows
        dataframes.append(batch.to_pandas())
    return row_count, dataframes


async def resolve_ws_endpoint(
    auth_mode: str | None = None,
    timeout: float | None = None,
    use_tls: bool | None = None,
    host: str | None = None,
    port: int | None = None,
    username: str | None = None,
    password: str | None = None,
):
    auth_mode = auth_mode or AUTH_MODE
    timeout = timeout or TIMEOUT_SECS
    use_tls = USE_TLS if use_tls is None else use_tls
    host = host or HOST
    port = port or PORT
    username = username or USERNAME
    password = password or PASSWORD

    scheme_http = "https" if use_tls else "http"
    scheme_ws = "wss" if use_tls else "ws"

    http_base = f"{scheme_http}://{host}:{port}"
    ws_base = f"{scheme_ws}://{host}:{port}/api/ws/sql"

    if auth_mode == "basic":
        headers = build_basic_auth_header(username, password)
        ws_url = ws_base
    elif auth_mode == "ticket":
        ticket = await fetch_ws_ticket(http_base, username, password, timeout)
        headers = None
        ws_url = f"{ws_base}?ticket={ticket}"
    else:
        raise ValueError("AUTH_MODE must be 'basic' or 'ticket'")

    return ws_url, headers, timeout


async def run_sql_dataframe(
    sql: str,
    *,
    auth_mode: str | None = None,
    timeout: float | None = None,
    ping_interval: float | None = None,
    ping_timeout: float | None = None,
    verbose: bool = True,
):
    ws_url, headers, timeout = await resolve_ws_endpoint(auth_mode=auth_mode, timeout=timeout)
    ping_interval = WS_PING_INTERVAL_SECS if ping_interval is None else ping_interval
    ping_timeout = WS_PING_TIMEOUT_SECS if ping_timeout is None else ping_timeout
    request_id = str(uuid.uuid4())
    total_rows = 0
    dataframes = []

    if verbose:
        print(f"Connecting: {ws_url}")

    async with websockets.connect(
        ws_url,
        additional_headers=headers,
        open_timeout=timeout,
        ping_interval=ping_interval,
        ping_timeout=ping_timeout,
    ) as ws:
        await ws.send(json.dumps({
            "type": "run_sql",
            "sql": sql,
            "request_id": request_id,
        }))

        if verbose:
            print(f"Sent run_sql request_id={request_id}")

        while True:
            frame = await ws.recv()

            if isinstance(frame, str):
                event = json.loads(frame)
                event_type = event.get("type")

                if verbose:
                    print("text event:", event)

                if event_type == "error":
                    raise RuntimeError(event.get("message", "unknown websocket error"))
                if event_type == "done":
                    break
            else:
                batch_rows, batch_frames = decode_arrow_stream_to_dataframes(bytes(frame))
                total_rows += batch_rows
                dataframes.extend(batch_frames)

    if verbose:
        print(f"Decoded rows from binary stream: {total_rows}")

    if not dataframes:
        return pd.DataFrame()

    return pd.concat(dataframes, ignore_index=True)


async def run_sql(sql: str, **kwargs):
    """Convenience alias for notebook cells: returns a pandas DataFrame."""
    return await run_sql_dataframe(sql, **kwargs)

In [106]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql(SQL)
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=a6c69921-57bf-4725-bd7e-e38880254578
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578'}
text event: {'type': 'chunk', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578', 'batch_index': 0, 'row_count': 1}
text event: {'type': 'done', 'request_id': 'a6c69921-57bf-4725-bd7e-e38880254578', 'total_rows': 1}
Decoded rows from binary stream: 1


,Int64(1)
0,1


In [130]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("CREATE ATLAS TABLE example LOCATION 'atlas-example'")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=bd9c5930-9641-4058-b3c7-7106cb1ead68
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'bd9c5930-9641-4058-b3c7-7106cb1ead68'}
text event: {'type': 'done', 'request_id': 'bd9c5930-9641-4058-b3c7-7106cb1ead68', 'total_rows': 0}
Decoded rows from binary stream: 0


""


In [147]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DESCRIBE example")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=fb173f4e-3458-43f0-b468-4ff8ce48d3e7
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'fb173f4e-3458-43f0-b468-4ff8ce48d3e7'}
text event: {'type': 'chunk', 'request_id': 'fb173f4e-3458-43f0-b468-4ff8ce48d3e7', 'batch_index': 0, 'row_count': 315}
text event: {'type': 'done', 'request_id': 'fb173f4e-3458-43f0-b468-4ff8ce48d3e7', 'total_rows': 315}
Decoded rows from binary stream: 315


,column_name,data_type,is_nullable
0,__entry_key,Utf8,YES
1,DATA_TYPE,Utf8,YES
2,DATA_TYPE.conventions,Utf8,YES
3,DATA_TYPE.long_name,Utf8,YES
4,DATA_TYPE._FillValue,Utf8,YES
...,...,...,...
310,references,Utf8,YES
311,featureType,Utf8,YES
312,institution,Utf8,YES
313,history,Utf8,YES


In [145]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("INGEST INTO ATLAS example ON PARTITION '2024-01-03' FROM 'aoml/**/*.nc' WITH netcdf")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=32dffda4-6c7e-476b-a328-6e500bd13b3d
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d'}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 0, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 1, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 2, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 3, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 4, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e500bd13b3d', 'batch_index': 5, 'row_count': 1}
text event: {'type': 'chunk', 'request_id': '32dffda4-6c7e-476b-a328-6e

,partition_name,dataset_name,status,message
0,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
1,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
2,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
3,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
4,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
...,...,...,...,...
9044,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
9045,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
9046,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted
9047,2024-01-03,/Users/robinkooyman/git/beacon/data/datasets/a...,ok,inserted


In [129]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("DROP TABLE example ")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=64f6f45e-8dc3-4b39-9e34-9977c7636594
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': '64f6f45e-8dc3-4b39-9e34-9977c7636594'}
text event: {'type': 'done', 'request_id': '64f6f45e-8dc3-4b39-9e34-9977c7636594', 'total_rows': 0}
Decoded rows from binary stream: 0


""


In [163]:
# Example usage: add more cells like this and call run_sql(...)
df = await run_sql("SELECT JULD, TEMP FROM example")
df

Connecting: ws://127.0.0.1:5001/api/ws/sql
Sent run_sql request_id=a1cbf864-e9c4-4920-8123-cd3089e132f5
text event: {'type': 'ready'}
text event: {'type': 'accepted', 'request_id': 'a1cbf864-e9c4-4920-8123-cd3089e132f5'}
text event: {'type': 'error', 'request_id': 'a1cbf864-e9c4-4920-8123-cd3089e132f5', 'message': 'Execution error: Failed to convert dataset 0 to Arrow stream: null values are not supported in AtlasArrowCompat'}


RuntimeError: Execution error: Failed to convert dataset 0 to Arrow stream: null values are not supported in AtlasArrowCompat